In [ ]:
# ============================================================
# BLOCK 0 — IMPORTS
# ============================================================
import numpy as np
from sklearn.metrics import f1_score, classification_report
from statsmodels.stats.contingency_tables import mcnemar


# ============================================================
# BLOCK 1 — LOAD PREDICTIONS + TRUE LABELS
# ============================================================
y_true_xlmr  = np.load('y_true_xlmr_ft_bi.npy')
y_pred_xlmr  = np.load('y_pred_xlmr_ft_bi.npy')

y_true_mbert = np.load('y_true_mbert_ft_bi.npy')
y_pred_mbert = np.load('y_pred_mbert_ft_bi.npy')


# ============================================================
# BLOCK 2 — CLASSIFICATION REPORTS
# ============================================================
y_true = y_true_xlmr

print("=== XLM-R ===")
print(classification_report(
    y_true,
    y_pred_xlmr,
    target_names=['No Distortion', 'Distortion']
))

print("=== mBERT ===")
print(classification_report(
    y_true,
    y_pred_mbert,
    target_names=['No Distortion', 'Distortion']
))


# ============================================================
# BLOCK 3 — COMPUTE CORRECT/INCORRECT PREDICTIONS
# ============================================================
xlmr_correct  = (y_pred_xlmr  == y_true).astype(int)
mbert_correct = (y_pred_mbert == y_true).astype(int)


# ============================================================
# BLOCK 4 — BUILD MCNEMAR CONTINGENCY TABLE
# ============================================================
both_correct        = np.sum((xlmr_correct == 1) & (mbert_correct == 1))
xlmr_only_correct   = np.sum((xlmr_correct == 1) & (mbert_correct == 0))
mbert_only_correct  = np.sum((xlmr_correct == 0) & (mbert_correct == 1))
both_wrong          = np.sum((xlmr_correct == 0) & (mbert_correct == 0))

contingency_table = np.array([
    [both_correct,       xlmr_only_correct],
    [mbert_only_correct, both_wrong       ]
])


# ============================================================
# BLOCK 5 — DISPLAY CONTINGENCY TABLE
# ============================================================
print("\n=== McNemar Contingency Table ===")
print(f"                  mBERT Correct  mBERT Wrong")
print(f"XLM-R Correct     {both_correct:<15} {xlmr_only_correct}")
print(f"XLM-R Wrong       {mbert_only_correct:<15} {both_wrong}")


# ============================================================
# BLOCK 6 — MCNEMAR'S TEST
# ============================================================
result = mcnemar(contingency_table, exact=False, correction=True)

print("\n=== McNemar's Test Result ===")
print(f"Statistic : {result.statistic:.4f}")
print(f"p-value   : {result.pvalue:.4f}")


# ============================================================
# BLOCK 7 — INTERPRETATION OF RESULTS
# ============================================================
if result.pvalue < 0.05:
    better = "XLM-R" if xlmr_only_correct > mbert_only_correct else "mBERT"
    print(f"\nConclusion: The difference is statistically significant (p < 0.05).")
    print(f"            {better} performs significantly better.")
else:
    print(f"\nConclusion: No statistically significant difference found (p >= 0.05).")